In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt

# Sample data — score from 0 (bad grammar) to 1 (perfect)
sentences = [
    "He go to school yesterday",          # 0.2
    "I has a book",                       # 0.3
    "They are playing football",          # 0.9
    "She readed the book",                # 0.4
    "We have gone to the park",           # 1.0
    "He eating apple",                    # 0.3
    "This is good example",               # 0.5
    "I am reading a book",                # 0.95,
]

scores = [0.2, 0.3, 0.9, 0.4, 1.0, 0.3, 0.5, 0.95]

# Tokenization
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, padding='post', maxlen=6)

print(padded)


model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=100, output_dim=16, input_length=6),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # fluency score from 0 to 1
])

model.compile(loss='mse', optimizer='adam', metrics=['mae'])
model.summary()


# Train the model
history = model.fit(padded, np.array(scores), epochs=100, verbose=0)

# Plot training
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['mae'], label='mae')
plt.legend()
plt.show()

# Function to score fluency of a sentence
def score_fluency(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    pad = pad_sequences(seq, maxlen=6, padding='post')
    score = model.predict(pad)[0][0]
    return round(score, 2)

# Try it out
print(score_fluency("He went to the park yesterday"))    # Should be high
print(score_fluency("She go home"))                      # Should be low



ModuleNotFoundError: No module named 'tensorflow'